###Working with timestamp
[Timestamp functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#date-and-timestamp-functions)


###1. How Spark stores the timestamp
1. Timestamp is internally stored as 12 byte integer known as INT96
2. Timestamp is made up of 7 fields
    1. Year
    2. Month
    3. Day
    4. Hour
    5. Minute
    6. Second
        * Up to 6 decimal places
        * Microsecond precision
    7. Timezone

##2. Requirement
You are given the below dataframes

In [0]:
data_list_1 = [(1, "2022-05-18T10:30:30.0000"), (2, "2022-05-19T11:30:10.0000")]
data_list_2 = [(1, "18-05-2022 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]
data_list_3 = [(1, "2022-05-18 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]

df_1 = (spark.createDataFrame(data_list_1).toDF("id", "string_time"))
df_2 = (spark.createDataFrame(data_list_2).toDF("id", "string_time"))
df_3 = (spark.createDataFrame(data_list_3).toDF("id", "string_time"))

2.1 covert the df_1 to timestamp

In [0]:
from pyspark.sql import functions as F
df_1.withColumn('string_time', F.to_timestamp(F.col('string_time'), "yyyy-MM-dd'T'HH:mm:ss.SSSS")).show()
df_2.withColumn('string_time', F.to_timestamp(F.col('string_time'), "dd-MM-yyyy HH:mm:ss.SSSS")).show()
df_3.withColumn('string_time', F.coalesce(F.try_to_timestamp(F.col('string_time')),F.to_timestamp(F.col('string_time'),"dd-MM-yyyy HH:mm:ss.SSSS"))).show()

### 3. Timezone information
1. A timestamp without timezone information is incomplete.
2. Spark offers two data types for timestamp
    1. TIMESTAMP
    2. TIMESTAMP_NTZ
3. For TIMESTAMP, Spark assumes session timezone as the default when timezone is not specified
4. Session timezone is specified as spark.sql.session.timeZone




%md
3.1 What is your default session timezone?

In [0]:
spark.conf.get('spark.sql.session.timeZone')

In [0]:
spark.conf.set('spark.sql.session.timeZone','Europe/Amsterdam')

###4. Working with NTZ data

4.1 Load machine-events-no-tz.csv file and show the data

In [0]:
from pyspark.sql.types import StringType, IntegerType, TimestampType, StructType, StructField
schema=StructType([
                    StructField("component", StringType()),
                    StructField("event_time", StringType()),
                    StructField("reading", StringType())
])
machine_eve_df=(
                    spark.read
                            .format("csv")
                            .option("header", True)
                            .option("schema", schema)
                            .load("/Volumes/dev/spark_db/datasets/spark_programming/data/machine-events-no-tz.csv")    
)

In [0]:
machine_eve_df.display()